## Exercise 1: Set Up the Testing Environment

In this exercise, you install the `pytest` library and review the transformation module you will be testing. The module processes raw order records through three stages: **filtering** invalid entries, **normalizing** status codes, and **computing** tax-inclusive totals.

Run the cells in this exercise — they are fully provided. Your challenge begins in Exercise 2.

In [ ]:
# ✅ Run this cell — installs pytest and creates the lab working directory
%pip install pytest
import os
os.makedirs('/tmp/lab12', exist_ok=True)

### ✅ Provided: Create the transformation module

Run the cell below to write the `transforms.py` module to `/tmp/lab12/`. The module exposes four functions:

| Function | Description |
|---|---|
| `filter_valid_orders(df)` | Removes rows where `order_id` or `customer_id` is null |
| `normalize_status(df)` | Converts `status` to lowercase and strips whitespace |
| `calculate_total_with_tax(df, tax_rate=0.21)` | Adds a `total_with_tax` column |
| `run_order_pipeline(df, tax_rate=0.21)` | Runs all three transforms in sequence |

Review the code carefully — you will be writing tests for each function in the next exercise.

In [ ]:
%%writefile /tmp/lab12/transforms.py
import pandas as pd


def filter_valid_orders(df):
    """Remove orders where order_id or customer_id is null."""
    return df.dropna(subset=['order_id', 'customer_id'])


def normalize_status(df):
    """Normalize status values to lowercase and strip whitespace."""
    df = df.copy()
    df['status'] = df['status'].str.lower().str.strip()
    return df


def calculate_total_with_tax(df, tax_rate=0.21):
    """Add a total_with_tax column based on the given tax rate."""
    df = df.copy()
    df['total_with_tax'] = (df['total'] * (1 + tax_rate)).round(2)
    return df


def run_order_pipeline(df, tax_rate=0.21):
    """Run the full order cleaning pipeline."""
    df = filter_valid_orders(df)
    df = normalize_status(df)
    df = calculate_total_with_tax(df, tax_rate)
    return df


### ✅ Provided: Verify the module loads correctly

Run the cell below to import the module and confirm the functions are accessible before writing any tests.

In [ ]:
# ✅ Run this cell — verifies the module imports correctly
import sys, os
os.makedirs('/tmp/lab12', exist_ok=True)
sys.path.insert(0, '/tmp/lab12')

from transforms import filter_valid_orders, normalize_status, calculate_total_with_tax, run_order_pipeline

print("✅ transforms.py loaded successfully.")
print("   Functions available:", [f.__name__ for f in [filter_valid_orders, normalize_status, calculate_total_with_tax, run_order_pipeline]])

## Exercise 2: Write Unit Tests

In this exercise, you write **unit tests** for each of the three transformation functions using pytest.

Use the sample dataset below as your pytest fixture:

| order_id | customer_id | status | total |
|---|---|---|---|
| 1001 | C001 | Completed | 150.00 |
| 1002 | C002 | PENDING | 200.50 |
| `null` | C003 | cancelled | 75.00 |
| 1004 | `null` | completed | 300.00 |
| 1005 | C005 | `  Pending  ` | 125.75 |

After filtering, **3 rows** remain (rows for `order_id` 1001, 1002, and 1005).

> 🤖 **Genie Code:** Ask *"Help me write pytest unit tests for pandas functions called filter_valid_orders, normalize_status, and calculate_total_with_tax. Include a fixture that returns a sample DataFrame."*

Fill in all `# TODO` blocks in the cell below, then run it to write the test file.

In [ ]:
%%writefile /tmp/lab12/test_transforms.py
import pytest
import pandas as pd
import sys
sys.path.insert(0, '/tmp/lab12')
from transforms import filter_valid_orders, normalize_status, calculate_total_with_tax


@pytest.fixture
def sample_orders():
    return pd.DataFrame({
        'order_id':    [1001,        1002,      None,        1004,        1005        ],
        'customer_id': ['C001',      'C002',    'C003',      None,        'C005'      ],
        'status':      ['Completed', 'PENDING', 'cancelled', 'completed', '  Pending  '],
        'total':       [150.00,      200.50,    75.00,       300.00,      125.75      ],
    })


# ── Tests for filter_valid_orders ─────────────────────────────────────────────

def test_filter_removes_null_order_id(sample_orders):
    result = filter_valid_orders(sample_orders)
    assert result['order_id'].isnull().sum() == 0


def test_filter_removes_null_customer_id(sample_orders):
    result = filter_valid_orders(sample_orders)
    assert result['customer_id'].isnull().sum() == 0


def test_filter_retains_correct_row_count(sample_orders):
    result = filter_valid_orders(sample_orders)
    assert len(result) == 3


# ── Tests for normalize_status ────────────────────────────────────────────────

def test_normalize_status_lowercases_values(sample_orders):
    result = normalize_status(sample_orders)
    assert all(result['status'] == result['status'].str.lower())


def test_normalize_status_strips_whitespace(sample_orders):
    result = normalize_status(sample_orders)
    assert all(result['status'] == result['status'].str.strip())


# ── Tests for calculate_total_with_tax ────────────────────────────────────────

def test_calculate_total_with_tax_adds_column(sample_orders):
    result = calculate_total_with_tax(sample_orders)
    assert 'total_with_tax' in result.columns


def test_calculate_total_with_tax_default_rate(sample_orders):
    result = calculate_total_with_tax(sample_orders)
    assert result.iloc[0]['total_with_tax'] == round(150.00 * 1.21, 2)


def test_calculate_total_with_tax_custom_rate(sample_orders):
    result = calculate_total_with_tax(sample_orders, tax_rate=0.10)
    assert result.iloc[1]['total_with_tax'] == round(200.50 * 1.10, 2)


### ✅ Run the unit tests

Run the cell below to execute your tests with pytest. **All 8 tests should pass.** If any fail, read the error output carefully, fix the failing test logic, re-run the `%%writefile` cell, then run the cell below again.

In [ ]:
# ✅ Run this cell — executes all unit tests
import sys
sys.path.insert(0, '/tmp/lab12')
import pytest

retcode = pytest.main(['/tmp/lab12/test_transforms.py', '-v', '-p', 'no:cacheprovider'])
assert retcode == 0, "❌ Unit tests failed. Review the output above and fix the failing tests."